In [ ]:
from pathlib import Path
import gzip
import re
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.io import mmwrite
from rich.console import Console

console = Console()
sc.set_figure_params(dpi=160, vector_friendly=False, color_map='viridis_r')


In [ ]:
PROJECT_DIR = next(
    root for root in [Path.cwd(), *Path.cwd().parents]
    if (root / 'function' / 'SpatialPyALab').is_dir()
)
DATA_ROOT = PROJECT_DIR  # Change to Path('/path/to/Project_ICC') if needed.

DATA_DIR = DATA_ROOT / 'Data/ST/Samples'
PATIENT_INFO = DATA_ROOT / 'Table/Patient_Info.csv'
FIG_DIR = DATA_ROOT / 'Figure/ST/Samples'
MERGED_FILE = DATA_ROOT / 'Data/ST/CellData/0_Allsample_addCelltype_cellSegment.h5ad'
SEURAT_DIR = DATA_ROOT / 'Data/ST/CellData/create_seurat_files'

sys.path.insert(0, str(PROJECT_DIR / 'function'))
import SpatialPyALab as lab


In [ ]:
def parse_sampleid(value):
    match = re.match(r'^Pt(\d+)-(\d+)([a-zA-Z])?$', str(value).strip())
    if match is None:
        return (10**9, 10**9, 10**9)
    patient, region, suffix = match.groups()
    suffix_rank = 0 if suffix is None else ord(suffix.lower()) - ord('a') + 1
    return (int(patient), int(region), suffix_rank)

patient_info = pd.read_csv(PATIENT_INFO)
sample_metadata = patient_info.dropna(subset=['SampleName', 'SampleID']).copy()
if sample_metadata['SampleName'].duplicated().any():
    raise ValueError('Patient_Info.csv must contain one metadata row per SampleName.')

sample_order = sample_metadata.copy()
sample_order[['patient_order', 'region_order', 'suffix_order']] = (
    sample_order['SampleID'].apply(lambda value: pd.Series(parse_sampleid(value)))
)
sample_order = sample_order.sort_values(
    ['patient_order', 'region_order', 'suffix_order', 'SampleID', 'SampleName']
)
samples = sample_order['SampleName'].tolist()
name2id = sample_metadata.set_index('SampleName')['SampleID'].astype(str).to_dict()

# To process a subset, replace the sample list here.
# samples = ['Sample01']
console.print(f'Processing {len(samples)} samples.')


In [ ]:
for idx, sample_id in enumerate(samples, 1):
    console.print(f'[{idx}/{len(samples)}] Annotating {sample_id}')
    input_dir = DATA_DIR / sample_id
    spot_file = input_dir / 'ficture' / f'{sample_id}_anno_ficture.h5ad'
    cell_file = input_dir / 'preprocess/cell_segment.h5ad'
    cell_out = input_dir / 'cell_segment/Cell_Segment_addCelltype_adata.h5ad'

    spot_adata = sc.read_h5ad(spot_file)
    spot_adata.obs['CellType'] = spot_adata.obs['CellType_ficture']
    spot_to_cell = lab.annotation.spot_to_cellid(
        spot_adata=spot_adata,
        spatial_key='spatial_manual_fine',
        mask_key='staining_image_mask',
    ).rename(columns={'Spot_ID': 'barcode'})
    spot_to_cell['cell_id'] = spot_to_cell['cell_id'].astype(str)

    columns = [col for col in ['barcode', 'CellType'] if col in spot_adata.obs.columns]
    ficture_anno = spot_adata.obs[columns].copy()
    if 'barcode' not in ficture_anno.columns:
        ficture_anno['barcode'] = spot_adata.obs_names
    label_df = lab.annotation.add_ficture_result(
        spot_to_other=spot_to_cell,
        spot_id='barcode',
        segment_id='cell_id',
        ficture_anno=ficture_anno,
        barcode_col='barcode',
        threshold=0.5,
        min_label_spots=2,
        label_cols=['CellType'],
    )
    del spot_adata, ficture_anno, spot_to_cell

    cell_adata = sc.read_h5ad(cell_file)
    mu_scale = cell_adata.uns['mu_scale']['spatial_manual_fine']
    cell_adata.obsm['spatial_um'] = cell_adata.obsm['spatial'] / mu_scale
    cell_adata.obs['cell_id'] = cell_adata.obs['cell_ID_mask'].astype(str)
    label_lookup = label_df.set_index('cell_id', verify_integrity=True)
    cell_adata.obs['CellType'] = cell_adata.obs['cell_id'].map(label_lookup['CellType'])
    cell_adata = cell_adata[cell_adata.obs['cell_ID_mask'] != 0].copy()

    cell_out.parent.mkdir(parents=True, exist_ok=True)
    cell_adata.write_h5ad(cell_out)
    del cell_adata, label_df, label_lookup


In [ ]:
CT_COLORS = {
    'Bcell': '#ba55d3',
    'Endothelial': '#1E90FF',
    'Epithelial': '#9fa6ab',
    'Hepatocyte': '#639d63',
    'Fibroblast': '#BD8F8F',
    'Myeloid': '#FFD500',
    'Tcell': '#FF0000',
    'unassigned': '#e9e9e9',
}

for idx, sample_id in enumerate(samples, 1):
    console.print(f'[{idx}/{len(samples)}] Plotting {sample_id}')
    cell_file = DATA_DIR / sample_id / 'cell_segment/Cell_Segment_addCelltype_adata.h5ad'
    cell_adata = sc.read_h5ad(cell_file)
    show_name = name2id.get(sample_id, sample_id)
    sample_fig_dir = FIG_DIR / show_name / 'add_ficture'
    sample_fig_dir.mkdir(parents=True, exist_ok=True)

    display_labels = cell_adata.obs['CellType'].astype('string')
    cell_adata.obs['CellType'] = (
        display_labels.fillna('unassigned').replace('conflict', 'unassigned').astype('category')
    )
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    fig.suptitle(show_name, fontsize=14, y=1.05)
    sc.pl.spatial(
        cell_adata,
        color='CellType',
        basis='spatial_um',
        palette=CT_COLORS,
        spot_size=10,
        show=False,
        ax=ax,
    )
    ax.set_aspect('equal')
    ax.axis('off')
    fig.savefig(sample_fig_dir / 'CellType_spatial.pdf', bbox_inches='tight')
    plt.close(fig)
    del cell_adata, display_labels


In [ ]:
adatas = []
for sample_id in samples:
    console.print(f'Reading {sample_id}')
    cell_file = DATA_DIR / sample_id / 'cell_segment/Cell_Segment_addCelltype_adata.h5ad'
    cell_adata = sc.read_h5ad(cell_file)
    cell_adata.obs['SampleName'] = sample_id
    new_barcodes = (
        cell_adata.obs['SampleName'].astype(str).str.strip().str.replace(r'\s+', '_', regex=True)
        + '_'
        + cell_adata.obs['cell_id'].astype(str).str.strip().str.replace(r'\s+', '_', regex=True)
    )
    cell_adata.obs_names = pd.Index(new_barcodes, name=None)
    cell_adata.obs['barcode'] = cell_adata.obs_names.astype(str)

    # Per-sample images are not retained by the original concatenation.
    cell_adata.uns.clear()
    adatas.append(cell_adata)

adata_merged = ad.concat(adatas, join='outer', fill_value=0)
del adatas, cell_adata
if not adata_merged.obs_names.is_unique:
    raise ValueError('Merged cell barcodes must be unique.')

metadata = sample_metadata.set_index('SampleName')
shared_columns = adata_merged.obs.columns.intersection(metadata.columns)
adata_merged.obs = adata_merged.obs.drop(columns=shared_columns).join(
    metadata, on='SampleName', how='left', validate='many_to_one'
)

ensembl_pattern = r'^(ENSG|ENST|ENSP|ENS)\d+(\.\d+)?$'
keep_genes = ~adata_merged.var_names.astype(str).str.match(ensembl_pattern)
adata_merged = adata_merged[:, keep_genes].copy()
MERGED_FILE.parent.mkdir(parents=True, exist_ok=True)
adata_merged.write_h5ad(MERGED_FILE)
console.print(f'Saved {adata_merged.n_obs} cells and {adata_merged.n_vars} genes: {MERGED_FILE}')


In [ ]:
SEURAT_DIR.mkdir(parents=True, exist_ok=True)
X = adata_merged.X
if not sp.issparse(X):
    X = sp.csr_matrix(np.asarray(X))
X = X.tocsr().astype(np.float32)
if X.nnz > 0:
    X.data = np.clip(X.data, 0, None)

with gzip.open(SEURAT_DIR / 'counts.mtx.gz', 'wb', compresslevel=1) as handle:
    mmwrite(handle, X)
pd.Series(adata_merged.var_names).to_csv(
    SEURAT_DIR / 'genes.tsv', sep='\t', index=False, header=False
)
pd.Series(adata_merged.obs['barcode'].astype(str).values).to_csv(
    SEURAT_DIR / 'barcodes.tsv', sep='\t', index=False, header=False
)
adata_merged.obs.to_csv(SEURAT_DIR / 'metadata.csv')
del X
console.print(f'Exported files to {SEURAT_DIR}')
